## 1. What is Regularization in Machine Learning? Why is it needed?

**Answer:**

Regularization is a technique used to reduce overfitting in machine learning models. It adds a penalty to large model coefficients so that the model learns general patterns instead of memorizing the training data.

---

## 2. Difference between Ridge Regression (L2) and Lasso Regression (L1)

**Answer:**

Ridge Regression (L2) reduces the size of coefficients but does not make them exactly zero. Lasso Regression (L1) can reduce some coefficients to zero, which helps in feature selection and simplifies the model.

---

## 3. What is Cross-Validation and why is it important?

**Answer:**

Cross-validation is a technique used to evaluate model performance on different subsets of data. It provides a more reliable estimate of model accuracy and helps detect overfitting.

---

## 4. Explain the following cross-validation techniques

### a) K-Fold Cross-Validation

**Answer:**

The dataset is divided into K equal parts. The model is trained on K−1 parts and tested on the remaining part. This process is repeated K times and the average score is calculated.

### b) Stratified K-Fold Cross-Validation

**Answer:**

Stratified K-Fold ensures that each fold has a similar distribution of target values. It helps create balanced splits and improves evaluation reliability.

### c) Leave-One-Out Cross-Validation (LOOCV)

**Answer:**

In LOOCV, one data point is used for testing while all remaining points are used for training. This process repeats until every sample has been used once as a test sample.

### d) Time Series Split

**Answer:**

Time Series Split is used for time-dependent data. It preserves the order of observations by training on past data and testing on future data.

---

## 5. Why are tree-based models less sensitive to feature scaling?

**Answer:**

Tree-based models make decisions using feature split points rather than distance calculations. Because of this, the scale of features has very little impact on their performance, so feature scaling is usually not required.


In [272]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import LeaveOneOut
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler




## Load Dataset

In [273]:
df = pd.read_csv('Advanced_Regression_HousePrice_Dataset.csv')
df.head(10)

,property_id,sale_date,area_sqft,bedrooms,bathrooms,location_score,property_age,distance_city_km,near_school,near_metro,crime_rate_index,house_price_inr
0,200001,2014-01-01,2181,6,4,8.1,21,3.8,0,0,4.84,35154898
1,200002,2019-12-01,2383,5,4,5.3,28,10.9,1,1,2.89,26710893
2,200003,2016-10-01,1047,3,3,5.9,7,27.5,0,1,4.04,11216242
3,200004,2013-03-01,1753,4,3,7.0,27,12.1,0,0,3.28,21984310
4,200005,2013-07-01,1728,4,4,10.0,32,1.4,0,1,3.84,25080429
5,200006,2017-09-01,2555,5,3,3.2,25,20.6,0,0,5.63,19298164
6,200007,2017-03-01,2460,6,5,9.0,20,6.9,0,1,4.86,42510938
7,200008,2022-04-01,1880,3,3,6.3,19,10.6,1,1,2.52,19892683
8,200009,2019-02-01,1057,2,3,8.4,13,5.1,0,0,4.93,13052453
9,200010,2017-07-01,1022,4,3,5.3,59,16.5,1,1,4.90,9146947


**Intuition:**

The dataset is loaded and the first few rows are displayed. This helps verify that the data has been imported correctly.

## Check Dataset Shape

In [274]:
print(df.shape)

(3800, 12)


* The shape shows the number of rows and columns. It gives a quick overview of the dataset size.

## Dataset Information

In [275]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3800 entries, 0 to 3799
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   property_id       3800 non-null   int64  
 1   sale_date         3800 non-null   object 
 2   area_sqft         3800 non-null   int64  
 3   bedrooms          3800 non-null   int64  
 4   bathrooms         3800 non-null   int64  
 5   location_score    3800 non-null   float64
 6   property_age      3800 non-null   int64  
 7   distance_city_km  3800 non-null   float64
 8   near_school       3800 non-null   int64  
 9   near_metro        3800 non-null   int64  
 10  crime_rate_index  3800 non-null   float64
 11  house_price_inr   3800 non-null   int64  
dtypes: float64(3), int64(8), object(1)
memory usage: 356.4+ KB



* This displays column names, data types, and non-null values. It helps understand the structure of the data.

## Missing Values

In [276]:
df.isnull().sum()

property_id         0
sale_date           0
area_sqft           0
bedrooms            0
bathrooms           0
location_score      0
property_age        0
distance_city_km    0
near_school         0
near_metro          0
crime_rate_index    0
house_price_inr     0
dtype: int64

* Missing values are checked before model building. This helps identify columns that may need cleaning.

## Statistical Summary

In [277]:
df.describe()

,property_id,area_sqft,bedrooms,bathrooms,location_score,property_age,distance_city_km,near_school,near_metro,crime_rate_index,house_price_inr
count,3800.00000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3.800000e+03
mean,201900.50000,1716.925526,3.428158,2.916316,6.502237,22.537105,13.085132,0.548421,0.472895,4.242911,2.071940e+07
std,1097.10984,582.996559,1.356682,1.133540,1.766945,12.325740,6.537425,0.497715,0.499330,2.045371,8.707465e+06
min,200001.00000,500.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.500000,1.506126e+06
25%,200950.75000,1322.000000,2.000000,2.000000,5.300000,14.000000,8.500000,0.000000,0.000000,2.810000,1.446895e+07
50%,201900.50000,1700.500000,3.000000,3.000000,6.500000,20.000000,13.000000,1.000000,0.000000,4.220000,1.989180e+07
75%,202850.25000,2105.000000,4.000000,4.000000,7.700000,29.000000,17.500000,1.000000,1.000000,5.650000,2.596062e+07
max,203800.00000,3776.000000,7.000000,6.000000,10.000000,80.000000,38.700000,1.000000,1.000000,12.000000,5.930315e+07



* Summary statistics show the distribution of numerical columns. It includes mean, minimum, and maximum values.

## Separate Features and Target

In [278]:
X = df.drop("house_price_inr", axis=1)

y = df["house_price_inr"]

* The target variable is separated from the input features. This is required before training a machine learning model.

## Train Test Split 

In [279]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

* The dataset is divided into training and testing sets. This helps evaluate model performance on unseen data.

## Feature Scaling

In [280]:
X = X.select_dtypes(include=['int64', 'float64'])

print(X.dtypes)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


property_id           int64
area_sqft             int64
bedrooms              int64
bathrooms             int64
location_score      float64
property_age          int64
distance_city_km    float64
near_school           int64
near_metro            int64
crime_rate_index    float64
dtype: object


* Cross Validation checks the model many times using different parts of the data. This helps us know how well the model is likely to perform on new data.


In [281]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

* Scaling brings all features to a similar range. This helps many machine learning algorithms perform better.

## Ridge Regression

In [282]:
ridge = Ridge(alpha=1.0)

ridge.fit(X_train, y_train)

ridge_score = ridge.score(X_test, y_test)

print(ridge_score)

0.9187014000846911


* Ridge Regression reduces overfitting by adding a penalty term. It helps improve model stability.

## Lasso Regression

In [283]:
lasso = Lasso(alpha=0.1)

lasso.fit(X_train, y_train)

lasso_score = lasso.score(X_test, y_test)

print(lasso_score)

0.918718098389661


* Lasso Regression applies regularization and reduces the effect of less important features. This can make the model simpler.

## Ridge Cross Validation

In [284]:
scores = cross_val_score(
    ridge,
    X_train,
    y_train,
    cv=5
)

print(scores.mean())

0.915279011750337


* Cross Validation checks the model on different parts of the training data. This helps get a more reliable performance score and reduces the chance of overfitting.


 ## K Fold Cross Validation

In [285]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    ridge,
    X_train,
    y_train,
    cv=kf
)

print(scores.mean())

0.9152738846655337


* The dataset is divided into several folds. Each fold gets a chance to be used for testing.

## Leave-One-Out Cross-Validation (LOOCV)


In [286]:
loo = LeaveOneOut()

scores = cross_val_score(
    ridge,
    X_train[:100],
    y_train[:100],
    cv=loo
)

print(scores.mean())

nan


c:\Users\jines\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
c:\Users\jines\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
c:\Users\jines\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
c:\Users\jines\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
c:\Users\jines\AppData\Local\Programs\Python\Python3

* One observation is used for testing and the rest for training. This process is repeated for all observations.

## Decision Tree Regression

In [287]:
dt = DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)

dt.fit(X_train, y_train)

dt_score = dt.score(X_test, y_test)

print(dt_score)

0.8836129597333776


* Decision Tree learns patterns by creating decision rules. It predicts values based on those learned rules.

## Random Forest Regression

In [288]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

rf_score = rf.score(X_test, y_test)

print(rf_score)

0.9265341731179251


* Random Forest combines multiple decision trees. This usually improves prediction accuracy and reduces overfitting.

## Support Vector Regression

In [289]:
svr_linear = SVR(kernel="linear")

svr_linear.fit(X_train, y_train)

linear_score = svr_linear.score(X_test, y_test)

print(linear_score)

-0.0019136011705465084


* Linear SVR is useful when the relationship between features and target is mostly linear. It tries to find the best fitting boundary.

## RBF SVR

In [290]:
svr_rbf = SVR(kernel="rbf")

svr_rbf.fit(X_train, y_train)

rbf_score = svr_rbf.score(X_test, y_test)

print(rbf_score)

-0.0029441615077301364


* RBF kernel can capture complex and non-linear relationships. It is useful when the data pattern is not linear.

## Model Evaluation

In [291]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

pred = rf.predict(X_test)

print("MAE :", mean_absolute_error(y_test, pred))
print("MSE :", mean_squared_error(y_test, pred))
print("R2 :", r2_score(y_test, pred))

MAE : 1783928.8108684206
MSE : 5916582806947.105
R2 : 0.9265341731179251


* Evaluation metrics are used to measure prediction performance. They show how close the predictions are to actual values.

## Compare Models

In [292]:
results = pd.DataFrame({
    "Model":[
        "Ridge",
        "Lasso",
        "Decision Tree",
        "Random Forest",
        "Linear SVR",
        "RBF SVR"
    ],
    "R2 Score":[
        ridge_score,
        lasso_score,
        dt_score,
        rf_score,
        linear_score,
        rbf_score
    ]
})

results

,Model,R2 Score
0,Ridge,0.918701
1,Lasso,0.918718
2,Decision Tree,0.883613
3,Random Forest,0.926534
4,Linear SVR,-0.001914
5,RBF SVR,-0.002944


* The performance of all models is compared together. This helps identify the best performing model.

# Final Analysis

In [293]:
best_score = results["R2 Score"].max()

best_model = results.loc[
    results["R2 Score"].idxmax(),
    "Model"
]

print("Best Model :", best_model)

print("Best R2 Score :", best_score)

Best Model : Random Forest
Best R2 Score : 0.9265341731179251


* The model with the highest R² score is selected as the best model. It provides the most accurate predictions on the dataset.